# 04 — Anchors and rule book

- Anchor = evidence-backed control point.
- It is not the nearest player by itself.
- Strong anchor evidence combines:
  - ball/player path intersection;
  - speed change near the player;
  - team distance margin;
  - opponent pressure and density;
  - subsequent control.
- `D` remains appropriate when flight, contest, or reception is unresolved.
- No fixed D hold duration is prescribed.

In [ ]:
from pathlib import Path
import sys
import pandas as pd

ROOT = Path.cwd()
if not (ROOT / 'possession').exists(): ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
from possession.baseline_pipeline import ClosestTeamRuleModel
from possession.calculations.features import build_engineered_features
from possession.temporal_rules import load_rules

features = build_engineered_features(model=ClosestTeamRuleModel(), rules=load_rules())
anchor_columns = [
    'anchor_team', 'anchor_source', 'anchor_confidence_score',
    'closest_distance_m', 'team_distance_margin_m', 'candidate_pressure_m',
    'candidate_opponent_density_5m', 'ball_distance_travelled_recent_m',
    'landing_opponent_approach_speed_mps', 'controlled_reception_evidence_score',
]
display(features[[c for c in anchor_columns if c in features]].head())


## Findings visual parity — anchor casebook

These are the original data-selected anchor/scenario plots. Each plot shows why a frame is or is not anchorable: player-path geometry, ball kinematics, pressure/density, scenario context, and persistence. A nearest-player point alone is not promoted to an anchor.


In [ ]:
# Anchor visual setup: reuse the same evidence columns and plotting helpers as the findings notebook.
import numpy as np
import matplotlib.pyplot as plt
from possession.calculations.plotting import plot_trajectory_window, plot_timeline_window, plot_evidence_window
from possession.baseline_pipeline import slot_team_map
from games import asset_paths
team_by_slot = slot_team_map(asset_paths()['atd_meta'])


In [ ]:
# Purpose: plot data-selected examples for the main unresolved anchors and scenarios.
def _first_frame(mask, lo=PUBLIC_START + 75, hi=PUBLIC_END - 75):
    mask = pd.Series(mask, index=features.index).fillna(False).astype(bool)
    candidates = features.loc[mask & features.frame.between(lo, hi), 'frame']
    return int(candidates.iloc[0]) if not candidates.empty else None

case_masks = {
    'comfortable anchor': features.get('low_pressure_control', pd.Series(False, index=features.index)) & features.get('selected_anchor_team', pd.Series(index=features.index)).notna(),
    'pressured or contested': features.get('scenario_context', pd.Series('', index=features.index)).isin(['CONTESTED', 'PRESSURED_CONTROL']),
    'ball in flight / reception race': features.get('scenario_context', pd.Series('', index=features.index)).eq('BALL_IN_FLIGHT'),
    'genuine reception / transition': features.get('genuine_reception_anchor', pd.Series(False, index=features.index)) | features.get('transition_anchor', pd.Series(False, index=features.index)),
}
case_frames = {name: _first_frame(mask) for name, mask in case_masks.items()}
display(pd.DataFrame({'case': case_frames.keys(), 'representative_frame': case_frames.values()}))
for name, centre in case_frames.items():
    if centre is None:
        print(f'{name}: no data-selected example in the public window')
        continue
    fig, axes = plot_evidence_window(features, centre - 75, centre + 75, team_by_slot=team_by_slot, ground_truth=gt_by_frame)
    axes[0].set_title(f'{name} — frame {centre}; 3 seconds before and after')
    axes[1].axvline(centre, color='black', ls='--', lw=1.2, label=f'candidate frame {centre}')
    axes[1].legend(loc='upper left', bbox_to_anchor=(0, -0.30), ncol=3, fontsize=8)
    plt.show()
    cols = [c for c in ['frame','closest_team','closest_distance_m','team_distance_margin_m','candidate_pressure_m','candidate_opponent_density_5m','arrival_time_margin_s','ball_speed_derived_mps','scenario_context','selected_anchor_source','selected_anchor_confidence','genuine_reception_anchor','transition_anchor'] if c in features]
    display(features.loc[features.frame.between(centre-5, centre+5), cols])